In [ ]:
# TASK 4.1

In [ ]:
#1 install and import
!pip install -q google-genai pydantic

import json
import pandas as pd
from getpass import getpass
from google import genai
from google.genai import types

In [ ]:
SCHEMA = {   # same idea as the name / age / city example in class: properties, their types, and required
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

In [ ]:
# keys and client
API_KEY=getpass("enter your API key:")
client=genai.Client(api_key=API_KEY)
MODEL_NAME="gemini-3.5-flash"

enter your API key:··········


In [ ]:
# CRAFT PROMPT
SYSTEM_TEXT = """
CONTEXT: You work for a clinic chain. When a patient misses an appointment, front-desk staff phone them and
write a short note about what the patient said.
ROLE: You are a careful call-centre analyst who works out why the patient missed the appointment.
ACTION: Read the note and return four fields:
- root_cause: exactly ONE of forgot_or_no_reminder, distance_or_transport, cost_or_fee,
  illness_or_emergency, work_or_schedule, other.
- confidence: a number from 0 to 1 (lower it when the note is vague).
- summary: at most 25 words.
- recommended_action: one concrete next step for the clinic.
If the note is ambiguous or unclear, answer other rather than guess.
TONE: neutral, factual, respectful, no blame.
FORMAT: reply with JSON only, with exactly these four fields.
"""

In [ ]:
def analyse_note(note_text):
  prompt=f" callcentre-note:\n{note_text}"
  response=client.models.generate_content(model=MODEL_NAME,
                                          contents=prompt,
                                          config=types.GenerateContentConfig(system_instruction=SYSTEM_TEXT,
                                                                             temperature=0,
                                                                             max_output_tokens=1000,
                                                                             response_mime_type="application/json",
                                                                             response_schema=SCHEMA,
                                                                            thinking_config=types.ThinkingConfig(thinking_level="minimal") ,),)
  return json.loads(response.text)




In [ ]:
notes=pd.read_csv("followup_notes.csv")
candidates=[c for c in notes.columns if c not in ("note_id","appointment_id")]
NOTE_COL=max(candidates,key=lambda c: notes[c].astype(str).str.len().mean())

n001=notes.loc[notes["note_id"]=="N001",NOTE_COL].iloc[0]
print("note text column:", NOTE_COL)
print("note N001:",n001)

result=analyse_note(n001)
print(result)

note text column: note_text
note N001: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'Patient forgot the appointment because she did not receive a reminder message and requested to rebook for next week.', 'recommended_action': "Reschedule the appointment for next week and verify the patient's phone number and notification settings in the system."}


In [ ]:
# task 4.2

import json, time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google.genai import types

# the allowed words and the Pydantic checklist (now defined BEFORE the loop)
ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

API_CALLS = 1      # Task 4.1 used 1 call

def get_raw_reply(note_text):
    """Send ONE note and return Gemini's reply as text, exactly as returned."""
    global API_CALLS
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=f"Call-centre note:\n{note_text}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT, temperature=0, max_output_tokens=1000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),
        ),
    )
    API_CALLS += 1
    return response.text or ""

FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
COLUMNS = ["note_id", "appointment_id"] + FIELDS + ["validation_status", "raw_response"]

rows = []
for _, record in notes.head(10).iterrows():
    note_text = record[NOTE_COL]
    row = {col: None for col in COLUMNS}
    row["note_id"] = record["note_id"]
    row["appointment_id"] = record["appointment_id"]

    # blank note: no API call
    if pd.isna(note_text) or str(note_text).strip() == "":
        row.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank")
        rows.append(row)
        print(record["note_id"], "-> skipped_blank (no API call)")
        continue

    # ask Gemini (if the server is busy, record it and carry on)
    try:
        raw = get_raw_reply(note_text)
    except Exception as error:
        row["validation_status"] = "invalid"
        rows.append(row)
        print(record["note_id"], "-> API error:", str(error)[:80])
        time.sleep(13)
        continue

    row["raw_response"] = raw                       # the reply exactly as returned

    # the checklist
    data = None
    try:
        data = json.loads(raw)                      # check 1: valid JSON?
        insight = FollowupInsight(**data)           # check 2: all fields, confidence 0 to 1?
        if insight.root_cause not in ALLOWED:       # check 3: allowed word?
            raise ValueError("not an allowed root_cause")
        for f in FIELDS:
            row[f] = getattr(insight, f)
        row["validation_status"] = "valid"
    except (json.JSONDecodeError, ValidationError, TypeError, ValueError):
        row["validation_status"] = "invalid"        # keep it, never repair it
        if isinstance(data, dict):
            for f in FIELDS:
                if f in data:
                    row[f] = data[f]                # copy only the fields the reply really had
    rows.append(row)
    print(record["note_id"], "->", row["validation_status"], "|", row["root_cause"])
    time.sleep(13)

notes_analysed = pd.DataFrame(rows, columns=COLUMNS)
notes_analysed.to_csv("notes_analysed.csv", index=False)
print("\nsaved notes_analysed.csv:", notes_analysed.shape)
print(notes_analysed["validation_status"].value_counts().to_string())
print("API calls so far:", API_CALLS)

N001 -> valid | forgot_or_no_reminder
N002 -> valid | distance_or_transport
N003 -> valid | cost_or_fee
N004 -> skipped_blank (no API call)
N005 -> valid | illness_or_emergency
N006 -> valid | work_or_schedule
N007 -> valid | forgot_or_no_reminder
N008 -> valid | distance_or_transport
N009 -> valid | other
N010 -> valid | work_or_schedule

saved notes_analysed.csv: (10, 8)
validation_status
valid            9
skipped_blank    1
API calls so far: 10


In [ ]:
# Agreement with the expected root causes from the assignment (at least 7 of the 9 non-blank notes)
EXPECTED = {"N001": "forgot_or_no_reminder", "N002": "distance_or_transport", "N003": "cost_or_fee",
            "N005": "illness_or_emergency", "N006": "work_or_schedule", "N007": "forgot_or_no_reminder",
            "N008": "distance_or_transport", "N009": "other", "N010": "work_or_schedule"}
agree = sum(1 for _, r in notes_analysed.iterrows()
            if r["note_id"] in EXPECTED and r["root_cause"] == EXPECTED[r["note_id"]])
n004 = notes_analysed.loc[notes_analysed["note_id"] == "N004", "validation_status"]
print("N004 status:", n004.iloc[0] if len(n004) else "not found", "(expected skipped_blank)")
print("agreement with the expected causes:", agree, "of", len(EXPECTED), "(at least 7 of 9 is required)")
print(notes_analysed[["note_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))

N004 status: skipped_blank (expected skipped_blank)
agreement with the expected causes: 9 of 9 (at least 7 of 9 is required)
note_id            root_cause  confidence validation_status
   N001 forgot_or_no_reminder        0.95             valid
   N002 distance_or_transport        0.95             valid
   N003           cost_or_fee        0.95             valid
   N004                 other        0.00     skipped_blank
   N005  illness_or_emergency        0.95             valid
   N006      work_or_schedule        0.95             valid
   N007 forgot_or_no_reminder        0.95             valid
   N008 distance_or_transport        0.95             valid
   N009                 other        0.20             valid
   N010      work_or_schedule        0.95             valid


In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

# Define the Pydantic model for validation
class FollowupInsight(BaseModel):
    root_cause: Literal["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee", "illness_or_emergency", "work_or_schedule", "other"]
    confidence: float = Field(..., ge=0.0, le=1.0)
    summary: str = Field(..., max_length=200)
    recommended_action: str

# (a) two deliberately bad replies (no API call)    task 4.3
bad_replies = {
    "confidence = 1.7": {"root_cause": "forgot_or_no_reminder", "confidence": 1.7,
                         "summary": "Patient forgot the visit", "recommended_action": "Send an SMS reminder"},
    "recommended_action missing": {"root_cause": "other", "confidence": 0.5, "summary": "The note is unclear"},
}
for label, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
        print(label, "-> accepted (unexpected)")
    except ValidationError as error:
        for e in error.errors():
            print(label, "-> rejected | field:", e["loc"][0], "| message:", e["msg"])



confidence = 1.7 -> rejected | field: confidence | message: Input should be less than or equal to 1
recommended_action missing -> rejected | field: recommended_action | message: Field required


In [ ]:
import json, time
from google.genai import types

V1_SYSTEM = "Classify this note."          # V1 = plain prompt, same schema and settings
compare = []
for nid in ["N007", "N009"]:
    note_text = notes.loc[notes["note_id"] == nid, NOTE_COL].iloc[0]
    try:
        response = client.models.generate_content(
            model=MODEL_NAME, contents=f"Call-centre note:\n{note_text}",
            config=types.GenerateContentConfig(
                system_instruction=V1_SYSTEM, temperature=0, max_output_tokens=1000,
                response_mime_type="application/json", response_schema=SCHEMA,
                thinking_config=types.ThinkingConfig(thinking_level="minimal")))
        API_CALLS += 1
        try:
            v1_label = json.loads(response.text).get("root_cause")
        except json.JSONDecodeError:
            v1_label = (response.text or "")[:40]
    except Exception as error:
        v1_label = "API error"
    v2_label = notes_analysed.loc[notes_analysed["note_id"] == nid, "root_cause"].iloc[0]
    compare.append({"note": nid, "V1 root_cause": v1_label,
                    "V1 in allowed list?": v1_label in ALLOWED, "V2 root_cause": v2_label})
    time.sleep(13)

print(pd.DataFrame(compare).to_string(index=False))
print("\nTotal API calls in Tasks 4.1 to 4.3:", API_CALLS, "(expected 12 to 14, and below 25)")

note                                           V1 root_cause  V1 in allowed list?         V2 root_cause
N007 Customer forgot appointment due to missing SMS reminder                False forgot_or_no_reminder
N009                                     Unreachable Patient                False                 other

Total API calls in Tasks 4.1 to 4.3: 12 (expected 12 to 14, and below 25)


In [ ]:
import os , json
import pandas as pd

for name in ["notes_analysed.csv","appointments_clean.csv","clinics_enriched.csv","test_risk_scores.csv","rejected_rows.json"]:
  print(name,"found" if os.path.exists(name) else "MISSING")

appts = pd.read_csv("appointments_clean.csv")
notes_analysed = pd.read_csv("notes_analysed.csv")
merged = notes_analysed.merge(appts[["appointment_id", "reminder_sent"]], on="appointment_id", how="left")
print("merged rows:", len(merged))
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))
print("\nnotes classed forgot_or_no_reminder, by reminder_sent:")
print(merged.loc[merged["root_cause"] == "forgot_or_no_reminder", ["note_id", "reminder_sent"]].to_string(index=False))
print("\nnotes whose appointment had reminder_sent = No:", int((merged["reminder_sent"] == "No").sum()), "of", len(merged))



notes_analysed.csv found
appointments_clean.csv found
clinics_enriched.csv found
test_risk_scores.csv found
rejected_rows.json found
merged rows: 10
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1

notes classed forgot_or_no_reminder, by reminder_sent:
note_id reminder_sent
   N001            No
   N007            No

notes whose appointment had reminder_sent = No: 5 of 10


In [ ]:
import json
from sklearn.metrics import roc_auc_score

# Values you chose earlier. Edit them if yours differ.
MODEL = "logistic regression"
CV_AUC = 0.77            # Task 3.5: best cross-validated ROC-AUC
THRESHOLD = 0.35         # Task 3.5: the threshold you chose
REMINDER_ODDS = 0.31     # Task 3.2: np.exp(coef / scale) for reminder_sent
GITHUB_LINK = "https://github.com/JC-Ranjith-AI/careloop_no_show_dsai--"   # paste your real link here (or edit README.md on GitHub)

# numbers recomputed from your own files
raw_rows = len(pd.read_csv("appointments.csv")) if os.path.exists("appointments.csv") else 1020
n_rejected = len(json.load(open("rejected_rows.json"))) if os.path.exists("rejected_rows.json") else None
overall = appts["no_show"].mean() * 100
band = pd.cut(appts["lead_days"], [0, 7, 14, 30, 10**6], labels=["1-7", "8-14", "15-30", "31+"])
band_rate = appts.groupby(band, observed=True)["no_show"].mean() * 100
clin = appts.merge(pd.read_csv("clinics_enriched.csv"), on="clinic_id").groupby("clinic_name")["no_show"].agg(["size", "mean"])
clin = clin[clin["size"] >= 95]
top_clinic, top_rate = clin["mean"].idxmax(), clin["mean"].max() * 100
scores = pd.read_csv("test_risk_scores.csv")
test_auc = roc_auc_score(scores["no_show"], scores["risk_prob"])
tp = int(((scores["flag"] == 1) & (scores["no_show"] == 1)).sum())
flagged = int(scores["flag"].sum())
precision, recall = tp / flagged, tp / int(scores["no_show"].sum())
nonblank = int((merged["validation_status"] != "skipped_blank").sum())
forgot = merged[merged["root_cause"] == "forgot_or_no_reminder"]
forgot_no_reminder = int((forgot["reminder_sent"] == "No").sum())

readme = f"""# CareLoop: predicting and understanding appointment no-shows

Repository: {"https://github.com/JC-Ranjith-AI/careloop_no_show_dsai--"}

## Summary
1. **Part 1 (data):** {raw_rows:,} raw rows became {len(appts):,} clean rows ({n_rejected} rows were rejected when parsing), and {overall:.1f}% of appointments are no-shows.
2. **Part 2 (SQL):** the no-show rate rises with lead time, from {band_rate['1-7']:.1f}% (1-7 days) to {band_rate['31+']:.1f}% (31+ days); the highest clinic with at least 95 appointments is {top_clinic} at {top_rate:.1f}%.
3. **Part 3 (model):** {MODEL} had the best cross-validated ROC-AUC ({CV_AUC:.2f}) and scored {test_auc:.2f} on the test set; at threshold {THRESHOLD} it flags {flagged} of {len(scores)} test appointments, with recall {recall:.2f} and precision {precision:.2f}.
4. **Part 4 (GenAI):** of {nonblank} non-blank call notes, {len(forgot)} were classed forgot_or_no_reminder, and {forgot_no_reminder} of those {len(forgot)} belong to appointments with no reminder sent.

## Recommendation
The front desk should phone, the evening before, the appointments the model flags as high risk. {MODEL.capitalize()} was chosen because it had the highest cross-validated ROC-AUC ({CV_AUC:.2f}, and {test_auc:.2f} on the test set), and a threshold of {THRESHOLD} was chosen on training data because it gave the best F1; on the test set it flags {flagged} of {len(scores)} appointments and catches {recall:.0%} of the real no-shows, in a clinic chain where {overall:.1f}% of appointments are no-shows. Lead time is the clearest practical lever: appointments booked 31 or more days ahead are no-shows {band_rate['31+']:.1f}% of the time against {band_rate['1-7']:.1f}% within a week, so those should be confirmed first. The notes point the same way, because {forgot_no_reminder} of the {len(forgot)} notes about forgetting belong to appointments with no reminder. In the model, a reminder is associated with lower odds of a no-show (about {REMINDER_ODDS:.2f} times), but the data are observational, so a small trial should test this before the clinic relies on it. The final threshold should depend on how many follow-up calls the front desk can make in an evening.

## Limitations
1. Only 10 call notes were analysed, so they show examples of real reasons, not clinic-wide patterns.
2. The data are observational, so associations (for example reminders and no-shows) are not proof of cause.
3. The cleaning statistics (medians and outlier fences) were calculated on all rows before the train/test split, so the test set had a small indirect influence on the cleaning.
"""
with open("README.md", "w") as f:
    f.write(readme)
print(readme)

# CareLoop: predicting and understanding appointment no-shows

Repository: https://github.com/JC-Ranjith-AI/careloop_no_show_dsai--

## Summary
1. **Part 1 (data):** 1,020 raw rows became 990 clean rows (32 rows were rejected when parsing), and 22.0% of appointments are no-shows.
2. **Part 2 (SQL):** the no-show rate rises with lead time, from 16.2% (1-7 days) to 60.5% (31+ days); the highest clinic with at least 95 appointments is Hadapsar Care Centre at 27.3%.
3. **Part 3 (model):** logistic regression had the best cross-validated ROC-AUC (0.77) and scored 0.79 on the test set; at threshold 0.35 it flags 49 of 198 test appointments, with recall 0.57 and precision 0.57.
4. **Part 4 (GenAI):** of 9 non-blank call notes, 2 were classed forgot_or_no_reminder, and 2 of those 2 belong to appointments with no reminder sent.

## Recommendation
The front desk should phone, the evening before, the appointments the model flags as high risk. Logistic regression was chosen because it had the high